# Wind Generation Reliability Analysis

**Objective:** Determine how reliably wind power can meet UK electricity demand, and recommend how many MW of wind power can be **dependably** counted on.

**Data Source:** [BMRS Elexon API](https://bmrs.elexon.co.uk/api-documentation) — `FUELHH` dataset (actual half-hourly wind generation, fuelType = WIND)

**Why this matters:** Wind power is variable — it depends on weather. Grid operators need to know not just the *average* output, but the *minimum they can reliably count on*. Over-estimating reliable capacity risks blackouts; under-estimating wastes expensive backup generation.

**Approach:**
1. Fetch historical wind generation data (2025)
2. Understand the overall distribution (range, spread, shape)
3. Analyze seasonal patterns (which months produce more wind?)
4. Analyze daily patterns (does wind vary by hour?)
5. Use percentile analysis to recommend reliable capacity
6. Assess consecutive low-wind periods ("wind droughts")

## 1. Data Collection

We fetch actual wind generation from the FUELHH dataset for the year 2025. This gives us half-hourly measurements — approximately 17,500 data points per year — providing a statistically robust sample.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import requests
from datetime import datetime

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 11

BASE_URL = "https://data.elexon.co.uk/bmrs/api/v1"

def fetch_actual_generation(from_date: str, to_date: str) -> pd.DataFrame:
    """Fetch actual wind generation from the FUELHH dataset."""
    url = f"{BASE_URL}/datasets/FUELHH/stream"
    params = {"from": from_date, "to": to_date}
    
    response = requests.get(url, params=params, timeout=60)
    response.raise_for_status()
    data = response.json()
    
    if not data:
        return pd.DataFrame(columns=["startTime", "generation"])
    
    df = pd.DataFrame(data)
    if "fuelType" in df.columns:
        df = df[df["fuelType"].str.upper() == "WIND"]
    
    df["startTime"] = pd.to_datetime(df["startTime"], utc=True)
    df["generation"] = pd.to_numeric(df["generation"], errors="coerce")
    
    return df[["startTime", "generation"]].dropna().sort_values("startTime").reset_index(drop=True)

# Fetch data for 2025
print("Fetching wind generation data for 2025...")
actuals = fetch_actual_generation("2025-01-01", "2025-12-31")
print(f"Retrieved {len(actuals):,} data points")
print(f"Date range: {actuals['startTime'].min()} to {actuals['startTime'].max()}")
actuals.head()

## 2. Overall Statistical Profile

Before diving into patterns, let's understand the basic characteristics of wind generation: its range, central tendency, and spread. These fundamentals frame the entire reliability discussion.

In [ ]:
gen = actuals["generation"]

print("=" * 55)
print("  WIND GENERATION — DESCRIPTIVE STATISTICS")
print("=" * 55)
print(f"  Count:             {len(gen):,} half-hourly readings")
print(f"  ──────────────────────────────────────────")
print(f"  Mean:              {gen.mean():,.0f} MW")
print(f"  Median:            {gen.median():,.0f} MW")
print(f"  Std Deviation:     {gen.std():,.0f} MW")
print(f"  ──────────────────────────────────────────")
print(f"  Minimum:           {gen.min():,.0f} MW")
print(f"  Maximum:           {gen.max():,.0f} MW")
print(f"  Range:             {gen.max() - gen.min():,.0f} MW")
print(f"  ──────────────────────────────────────────")
print(f"  Coeff of Variation: {gen.std() / gen.mean() * 100:.1f}%")
print("=" * 55)

print(f"\n→ The coefficient of variation ({gen.std() / gen.mean() * 100:.1f}%) is very high.")
print(f"  For comparison, stable baseload sources (nuclear) have CV ~5-10%.")
print(f"  This confirms wind is a HIGHLY VARIABLE energy source.")
print(f"\n→ The range ({gen.min():,.0f} to {gen.max():,.0f} MW) shows that wind output")
print(f"  can swing from near-zero to maximum within the year.")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: Histogram of generation values
axes[0].hist(gen, bins=80, color="#3b82f6", edgecolor="white", alpha=0.85)
axes[0].axvline(gen.mean(), color="red", linestyle="--", linewidth=2, label=f"Mean = {gen.mean():,.0f} MW")
axes[0].axvline(gen.median(), color="orange", linestyle="--", linewidth=2, label=f"Median = {gen.median():,.0f} MW")
axes[0].set_xlabel("Wind Generation (MW)")
axes[0].set_ylabel("Count (half-hourly readings)")
axes[0].set_title("Distribution of Wind Generation")
axes[0].legend()

# Right: Box plot to show spread and outliers
bp = axes[1].boxplot(gen, vert=True, patch_artist=True,
                     boxprops=dict(facecolor="#3b82f6", alpha=0.6),
                     medianprops=dict(color="red", linewidth=2))
axes[1].set_ylabel("Wind Generation (MW)")
axes[1].set_title("Box Plot of Wind Generation")
axes[1].set_xticklabels(["All Data"])

plt.suptitle("Wind Generation Distribution — 2025", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

## 3. Percentile Analysis — The Foundation of Reliability

Percentiles are the key tool for reliability assessment. Here's the intuition:

- **P50** = the value exceeded 50% of the time (same as median) — using this means shortfalls half the time
- **P90** = the value exceeded 90% of the time — industry standard for "firm" capacity
- **P95** = exceeded 95% of the time — conservative estimate
- **P99** = exceeded 99% of the time — very conservative, near-minimum

The trade-off: higher percentile → more reliable → but lower MW value (you're being more cautious).

In [ ]:
# Calculate key percentiles
# Note: P90 "exceedance" means the value BELOW which only 10% of data falls
# So we use quantile(0.10) for P90 exceedance
percentile_levels = [1, 5, 10, 25, 50, 75, 90, 95, 99]
percentiles = {}

print("=" * 55)
print("  PERCENTILE ANALYSIS")
print("  (Exceedance: % of time generation is ABOVE this value)")
print("=" * 55)

for p in percentile_levels:
    # Exceedance percentile: P90 exceedance = quantile(0.10)
    # i.e., generation exceeds this value 90% of the time
    exceedance_value = gen.quantile((100 - p) / 100)
    percentiles[f"P{p}"] = exceedance_value
    marker = " ← INDUSTRY STANDARD" if p == 90 else ""
    print(f"  P{p:>2} exceedance:  {exceedance_value:>8,.0f} MW  (generation exceeds this {p}% of time){marker}")

print("=" * 55)
print(f"\n→ P90 exceedance = {percentiles['P90']:,.0f} MW")
print(f"  This means wind generation was above {percentiles['P90']:,.0f} MW for 90% of all half-hours.")
print(f"  This is the value most commonly used in capacity planning.")
print(f"\n→ P95 exceedance = {percentiles['P95']:,.0f} MW")
print(f"  A more conservative estimate: available 95% of the time.")

In [ ]:
# Visualize the CDF (Cumulative Distribution Function)
sorted_gen = np.sort(gen.values)
exceedance_prob = 1 - np.arange(1, len(sorted_gen) + 1) / len(sorted_gen)

fig, ax = plt.subplots(figsize=(14, 7))
ax.plot(sorted_gen, exceedance_prob * 100, color="#3b82f6", linewidth=2)
ax.fill_between(sorted_gen, exceedance_prob * 100, alpha=0.1, color="#3b82f6")

# Mark key percentiles
for p, color, ls in [(50, "gray", "--"), (90, "red", "--"), (95, "orange", ":"), (99, "purple", ":")]:
    val = percentiles[f"P{p}"]
    ax.axhline(p, color=color, linestyle=ls, alpha=0.5)
    ax.axvline(val, color=color, linestyle=ls, alpha=0.5)
    ax.annotate(f"P{p} = {val:,.0f} MW", xy=(val, p), fontsize=10, fontweight="bold",
               xytext=(val + gen.max()*0.03, p + 2), color=color,
               arrowprops=dict(arrowstyle="->", color=color, lw=1.5))

ax.set_xlabel("Wind Generation (MW)")
ax.set_ylabel("Exceedance Probability (%)")
ax.set_title("Wind Generation Duration Curve — How Often is Generation Above X MW?")
ax.set_ylim(-2, 102)
ax.set_xlim(left=0)
plt.tight_layout()
plt.show()

print("Reading this chart: Pick any MW value on the X-axis, go up to the curve,")
print("and read across to the Y-axis to see what % of the time generation exceeds that value.")

## 4. Seasonal (Monthly) Patterns

UK wind generation varies significantly by season. Winter months tend to be windier due to stronger Atlantic weather systems, while summer months are typically calmer. Understanding this seasonality is critical for reliability — we need to know the *weakest* months, not just the average.

In [ ]:
actuals["month"] = actuals["startTime"].dt.month
actuals["month_name"] = actuals["startTime"].dt.strftime("%b")

monthly = actuals.groupby("month")["generation"].agg(
    ["mean", "median", "std", "min", "max", "count"]
).round(0)

# Also calculate P90 exceedance per month
monthly["p90"] = actuals.groupby("month")["generation"].quantile(0.10).round(0)

month_names = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", 
               "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
monthly.index = [month_names[i-1] for i in monthly.index]

print("Monthly Wind Generation Statistics (MW):")
print("=" * 80)
print(monthly.to_string())
print("=" * 80)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

months = monthly.index
x = range(len(months))

# Left: Mean generation with min-max range
axes[0].bar(x, monthly["mean"], color="#3b82f6", edgecolor="white", alpha=0.8, label="Mean")
axes[0].plot(x, monthly["p90"], "ro-", linewidth=2, markersize=7, label="P90 exceedance")
axes[0].set_xticks(x)
axes[0].set_xticklabels(months)
axes[0].set_xlabel("Month")
axes[0].set_ylabel("Generation (MW)")
axes[0].set_title("Monthly Mean & P90 Wind Generation")
axes[0].legend()

# Right: Box plots per month
month_data = [actuals[actuals["month"] == m]["generation"].values 
              for m in sorted(actuals["month"].unique())]
bp = axes[1].boxplot(month_data, patch_artist=True, labels=months[:len(month_data)])
for patch in bp["boxes"]:
    patch.set_facecolor("#3b82f6")
    patch.set_alpha(0.6)
axes[1].set_xlabel("Month")
axes[1].set_ylabel("Generation (MW)")
axes[1].set_title("Monthly Generation Distribution")

plt.suptitle("Seasonal Wind Generation Patterns", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

if len(monthly) > 0:
    best_month = monthly["mean"].idxmax()
    worst_month = monthly["mean"].idxmin()
    print(f"\nWindiest month: {best_month} (mean = {monthly.loc[best_month, 'mean']:,.0f} MW)")
    print(f"Calmest month:  {worst_month} (mean = {monthly.loc[worst_month, 'mean']:,.0f} MW)")
    print(f"\nThe P90 line (red) shows the reliable baseline for each month.")
    print(f"For capacity planning, we must plan for the WEAKEST month's P90,")
    print(f"not the average — otherwise we'd have shortfalls during calm periods.")

## 5. Time-of-Day Patterns

Wind speed often varies through the day due to thermal effects. Let's check if UK wind generation shows a consistent daily pattern.

In [ ]:
actuals["hour"] = actuals["startTime"].dt.hour

hourly = actuals.groupby("hour")["generation"].agg(
    ["mean", "median", "min", "max", "std"]
).round(0)

# P90 exceedance by hour
hourly["p90"] = actuals.groupby("hour")["generation"].quantile(0.10).round(0)

fig, ax = plt.subplots(figsize=(14, 6))
hours = hourly.index

ax.fill_between(hours, hourly["min"], hourly["max"], alpha=0.1, color="#3b82f6", label="Min-Max range")
ax.plot(hours, hourly["mean"], "o-", color="#3b82f6", linewidth=2, markersize=6, label="Mean")
ax.plot(hours, hourly["median"], "s--", color="#22c55e", linewidth=2, markersize=5, label="Median")
ax.plot(hours, hourly["p90"], "^:", color="red", linewidth=2, markersize=5, label="P90 exceedance")

ax.set_xlabel("Hour of Day (UTC)")
ax.set_ylabel("Generation (MW)")
ax.set_title("Wind Generation by Time of Day")
ax.set_xticks(range(0, 24))
ax.legend()
plt.tight_layout()
plt.show()

best_hr = hourly["mean"].idxmax()
worst_hr = hourly["mean"].idxmin()
print(f"\nHighest average generation: {best_hr}:00 UTC ({hourly.loc[best_hr, 'mean']:,.0f} MW)")
print(f"Lowest average generation:  {worst_hr}:00 UTC ({hourly.loc[worst_hr, 'mean']:,.0f} MW)")
diurnal_range = hourly['mean'].max() - hourly['mean'].min()
print(f"\nDiurnal range: {diurnal_range:,.0f} MW ({diurnal_range/gen.mean()*100:.1f}% of overall mean)")
print(f"→ {'Significant' if diurnal_range/gen.mean()*100 > 10 else 'Modest'} daily variation.")

## 6. Wind Drought Analysis

One of the biggest risks with wind power is extended periods of low wind — "wind droughts." Even if the average is high, a weeklong calm period could cause serious supply issues.

Let's identify periods where wind generation dropped below a low threshold for consecutive hours.

In [ ]:
# Define "low wind" as below the P90 exceedance value
low_wind_threshold = percentiles["P90"]
print(f"Low wind threshold (P90 exceedance): {low_wind_threshold:,.0f} MW")

# Mark periods below threshold
actuals["below_threshold"] = actuals["generation"] < low_wind_threshold

# Find consecutive runs of low wind
actuals["low_wind_group"] = (~actuals["below_threshold"]).cumsum()
low_wind_runs = actuals[actuals["below_threshold"]].groupby("low_wind_group").agg(
    start=("startTime", "first"),
    end=("startTime", "last"),
    duration_hours=("startTime", lambda x: len(x) * 0.5),  # each reading = 30 min
    min_generation=("generation", "min"),
    mean_generation=("generation", "mean"),
).sort_values("duration_hours", ascending=False)

print(f"\nTotal low-wind events (below {low_wind_threshold:,.0f} MW): {len(low_wind_runs)}")
print(f"\nTop 10 longest low-wind periods:")
print("=" * 90)
for i, (_, row) in enumerate(low_wind_runs.head(10).iterrows()):
    print(f"  {i+1:>2}. {row['start'].strftime('%Y-%m-%d %H:%M')} → {row['end'].strftime('%Y-%m-%d %H:%M')}  "
          f"| Duration: {row['duration_hours']:>5.1f}h  "
          f"| Min: {row['min_generation']:>6,.0f} MW  "
          f"| Avg: {row['mean_generation']:>6,.0f} MW")
print("=" * 90)

if len(low_wind_runs) > 0:
    longest = low_wind_runs.iloc[0]
    print(f"\n→ Longest low-wind period: {longest['duration_hours']:.0f} hours ({longest['duration_hours']/24:.1f} days)")
    print(f"  Minimum generation during this period: {longest['min_generation']:,.0f} MW")
    print(f"  This represents the worst-case scenario that backup systems must cover.")

## 7. Time Series Overview

Let's visualize the full time series to see overall patterns and identify any anomalous periods.

In [ ]:
fig, ax = plt.subplots(figsize=(18, 6))

ax.plot(actuals["startTime"], actuals["generation"], linewidth=0.5, alpha=0.6, color="#3b82f6")

# Add rolling average (24h = 48 half-hours)
rolling_avg = actuals.set_index("startTime")["generation"].rolling("24h").mean()
ax.plot(rolling_avg.index, rolling_avg.values, linewidth=2, color="#ef4444", label="24h rolling average")

# Mark key levels
ax.axhline(gen.mean(), color="orange", linestyle="--", alpha=0.7, label=f"Mean = {gen.mean():,.0f} MW")
ax.axhline(percentiles["P90"], color="green", linestyle="--", alpha=0.7, label=f"P90 exceedance = {percentiles['P90']:,.0f} MW")

ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
ax.xaxis.set_major_locator(mdates.MonthLocator())
ax.set_xlabel("Date")
ax.set_ylabel("Wind Generation (MW)")
ax.set_title("UK Wind Generation Throughout 2025")
ax.legend(loc="upper right")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print("The blue trace shows raw half-hourly readings; the red line is a 24h moving average.")
print("Notice the high variability (blue spikes) compared to the smoother trend (red).")

## 8. Capacity Factor Analysis

The **capacity factor** tells us how effectively the installed wind capacity is being utilized. It's defined as:

$$\text{Capacity Factor} = \frac{\text{Actual Generation}}{\text{Maximum Possible Generation}} \times 100\%$$

We use the observed maximum as a proxy for installed capacity.

In [ ]:
installed_capacity_proxy = gen.max()  # Use observed max as proxy

avg_cf = (gen.mean() / installed_capacity_proxy) * 100
median_cf = (gen.median() / installed_capacity_proxy) * 100
min_cf = (gen.min() / installed_capacity_proxy) * 100

print("=" * 55)
print("  CAPACITY FACTOR ANALYSIS")
print("=" * 55)
print(f"  Estimated installed capacity:  {installed_capacity_proxy:,.0f} MW")
print(f"  ──────────────────────────────────────────")
print(f"  Average capacity factor:       {avg_cf:.1f}%")
print(f"  Median capacity factor:        {median_cf:.1f}%")
print(f"  Minimum capacity factor:       {min_cf:.1f}%")
print("=" * 55)
print(f"\n→ An average CF of {avg_cf:.0f}% is {'typical' if 25 <= avg_cf <= 45 else 'unusual'} for UK wind.")
print(f"  UK onshore wind typically achieves 25-30% CF, offshore 35-45%.")
print(f"  The blended national figure reflects the mix of both.")

## 9. Recommendation: Reliable Wind Power Capacity

Now we bring together all the analysis to answer the key question:

> **How many MW of wind power can we reliably expect to be available to meet electricity demand?**

### Framework for the Recommendation

The answer depends on the **risk tolerance** of the grid operator:

| Confidence Level | Percentile Used | Meaning | Use Case |
|---|---|---|---|
| Moderate (50%) | P50 exceedance | Available half the time | Financial planning |
| High (90%) | P90 exceedance | Available 90% of the time | **Capacity planning (standard)** |
| Very High (95%) | P95 exceedance | Available 95% of the time | Conservative planning |
| Near-certain (99%) | P99 exceedance | Available 99% of the time | Critical infrastructure |

In [ ]:
p50_val = percentiles["P50"]
p90_val = percentiles["P90"]
p95_val = percentiles["P95"]
p99_val = percentiles["P99"]

# Find the worst month's P90 for seasonal resilience
monthly_p90 = actuals.groupby("month")["generation"].quantile(0.10)
worst_month_p90 = monthly_p90.min()
worst_month_idx = monthly_p90.idxmin()
worst_month_name = month_names[worst_month_idx - 1] if worst_month_idx <= 12 else "Unknown"

print("╔" + "═" * 63 + "╗")
print("║          RELIABLE WIND CAPACITY RECOMMENDATION              ║")
print("╠" + "═" * 63 + "╣")
print(f"║                                                               ║")
print(f"║  RECOMMENDED RELIABLE CAPACITY:  {p90_val:>8,.0f} MW (P90 annual)   ║")
print(f"║                                                               ║")
print(f"║  Seasonal-adjusted (worst month): {worst_month_p90:>7,.0f} MW ({worst_month_name} P90) ║")
print(f"║                                                               ║")
print("╠" + "═" * 63 + "╣")
print(f"║  Supporting evidence:                                         ║")
print(f"║  • P90 exceedance: {p90_val:>7,.0f} MW (available 90% of time)    ║")
print(f"║  • P95 exceedance: {p95_val:>7,.0f} MW (available 95% of time)    ║")
print(f"║  • P99 exceedance: {p99_val:>7,.0f} MW (available 99% of time)    ║")
print(f"║  • Mean generation: {gen.mean():>6,.0f} MW (NOT reliable — 50% shortfall) ║")
print(f"║  • Worst month P90:  {worst_month_p90:>5,.0f} MW ({worst_month_name}: seasonal low)      ║")
print("╚" + "═" * 63 + "╝")

## 10. Reasoning & Conclusions

### Primary Recommendation

**We recommend using the P90 exceedance value as the reliable wind capacity figure.**

### Reasoning

1. **Why P90?** The P90 (90% exceedance) is the industry-standard metric for capacity planning in renewable energy. It represents the generation level that is exceeded 90% of the time. Using the mean (~P50) would result in supply shortfalls approximately 50% of the time, which is unacceptable for grid reliability.

2. **Why not P95 or P99?** While more conservative:
   - P95 and P99 yield significantly lower capacity values, meaning we'd need far more expensive backup capacity
   - P90 strikes the right balance between reliability and economic efficiency
   - Grid operators already maintain spinning reserves and demand response programs to cover the remaining 10% of cases

3. **Seasonal consideration:** The weakest month's P90 value may be more appropriate for year-round planning of firm capacity. The annual P90 includes windier winter months that inflate the average; using the worst month's P90 ensures reliability even during seasonally calm periods.

4. **Wind drought risk:** Our analysis identified extended low-wind periods lasting multiple days. Even the P90 value doesn't protect against these rare but impactful events. Grid operators need additional reserves (gas turbines, battery storage, interconnectors) to handle wind droughts.

### Key Caveats

- **One year of data is limited.** Multi-year analysis would capture year-to-year variability (some years are windier than others).
- **Wind capacity is growing.** New turbines being installed mean historical data may understate future generation.
- **Correlation with demand matters.** If low-wind periods coincide with high demand (e.g., cold winter anticyclones), the risk is compounded. A full reliability study requires joint analysis of wind supply AND electricity demand.
- **Geographic diversity helps.** National-level aggregation already smooths out local wind variations, but further geographic spread of wind farms would improve reliability.

### Summary

| Metric | Value | Interpretation |
|--------|-------|----------------|
| Mean generation | ~X MW | NOT reliable — shortfalls 50% of the time |
| P90 exceedance (annual) | ~Y MW | **Recommended reliable capacity** — available 90% of hours |
| P90 exceedance (worst month) | ~Z MW | More conservative — accounts for seasonal lows |
| Capacity factor | ~W% | Typical for UK wind mix (onshore + offshore) |
| Longest wind drought | ~N hours | Duration backup systems must cover |